# 12. Análisis del lenguaje con spaCy

De esta unidad dispones de los siguientes archivos:

- Los programas y archivos necesarios: [programas12-spa.zip](https://github.com/aoliverg/python/blob/master/programas12-spa/programas12-spa.zip)

---

# Instrucciones previas para la descarga/subida de archivos necesarios (`freeling-spa.txt`)

Para poder ejecutar las secciones de este notebook relacionadas con la descarga de archivos externos o diccionarios de FreeLing (Sección 4), es necesario disponer de los ficheros correspondientes en el directorio de trabajo.

A continuación dispones de las utilidades de código según cómo estés ejecutando el notebook:

### Opción A: Descarga automática desde Python (Recomendado para Colab o Local)

In [ ]:
import urllib.request
import os

# Descarga del diccionario de FreeLing necesario para la sección 4
url = "https://raw.githubusercontent.com/aoliverg/python/master/programas12-spa/freeling-spa.txt"
fichero_destino = "freeling-spa.txt"

if not os.path.exists(fichero_destino):
    print("Descargando 'freeling-spa.txt'...")
    try:
        urllib.request.urlretrieve(url, fichero_destino)
        print("¡Se ha descargado el archivo freeling-spa.txt correctamente!")
    except Exception as e:
        print(f"No se ha podido descargar el archivo automáticamente: {e}")
else:
    print("El archivo 'freeling-spa.txt' ya existe en el directorio local.")

### Opción B: Subir el archivo manualmente si estás en Google Colab

In [ ]:
# Solo necesario si estás ejecutando esto en Google Colab y quieres subir el archivo desde tu ordenador
try:
    from google.colab import files
    uploaded = files.upload()  # Selecciona 'freeling-spa.txt' o el ZIP de recursos desde tu disco duro
except ImportError:
    print("Esta celda solo se ejecuta en Google Colab.")

---

## 1. Introducción

### 1.1. ¿Qué es spaCy?

spaCy es un conjunto de herramientas de Procesamiento del Lenguaje Natural (PLN o en inglés NLP, de Natural Language Processing) diseñado específicamente para entornos de producción y la extracción de información a gran escala. Permite analizar textos en todos los niveles de análisis, desde la superficie (las palabras) hasta su estructura profunda (las relaciones gramaticales).

Sus características principales son:

- **Rendimiento y velocidad**: Está escrito en Cython desde cero con una gestión de memoria muy cuidada, lo que lo hace extremadamente rápido.
- **Precisión industrial**: Incorpora modelos de aprendizaje profundo (incluyendo transformadores como BERT) para lograr una gran precisión.
- **Multilingüe**: Soporta más de 75 idiomas y ofrece modelos preentrenados para muchos de ellos, incluidos el castellano y el catalán.

**La arquitectura de análisis**

Cuando pasamos un texto por spaCy, este atraviesa una "tubería" (_pipeline_) donde cada paso (como el tokenizador, el etiquetador gramatical o el parser sintáctico) añade una capa de información al objeto resultante. El usuario tiene un control total sobre este _pipeline_ y puede activar, desactivar o añadir los niveles de análisis necesarios según el proyecto.

La web oficial de spaCy es: [https://spacy.io/](https://spacy.io/). Dedica unos minutos a explorar la web y lee con atención la introducción al sistema.

### 1.2. ¿Cómo se instala spaCy?

La instalación es muy directa mediante pip. Antes de instalar nada, plantéate si quieres crear un entorni virtual de Python (recomendado).

**Paso 1: Instalar el paquete**

Abre el terminal o cónsola y escribe:

`pip install spacy`

Paso 2: Descargar los modelos de idioma

A diferencia de otras librerías de NLP, spaCy no incluye los modelos lingüísticos directamente en el paquete por defecto para no ocupar demasiado espacio. Debes descargar el modelo específico para el idioma y tamaño que quieras analizar. En spaCy, los modelos se descargan habitualmente por línea de comandos.

Por ejemplo, para descargar el modelo de tamaño medio del español. abre el terminal y ejecuta:

`python -m spacy download es_core_news_md`

(Nota: spaCy suele ofrecer diferentes tamaños de modelos como sm (small), md (medium), lg (large) o trf (transformer). Para empezar, el md es un buen equilibrio entre precisión y velocidad.) En algunos casos se puede necesitar una dependencia adicional: click. Escribe

`pip install click`

También es posible añadir a nuestros programas una función que verifique si el modelo está instalado y, si no, informar o intentar descargarlo. Podría ser así (programa-12.1-spa.py):

In [ ]:
import spacy
from spacy.cli import download  # Importamos la herramiena de descarga al principio

def carregar_nlp(nom_model='es_core_news_md'):
    try:
        print(f"--- Inicializando el modelo  ({nom_model})...")
        # exclude=['ner'] evita cargar el reconocedor de entidades si no lo necesitamos
        return spacy.load(nom_model, exclude=['ner'])
    except OSError:
        print(f"--- El modelo '{nom_model}' no se ha encontrado.")
        print("Descargándolo automáticamente. Esto puede tardar un poco...")
        
        # Descargamos el model directamente
        download(nom_model)
        
        # Volvemos a intentar cargarlo una vez descargado
        print(f"--- Carga completada. Inicializando ({nom_model})...")
        return spacy.load(nom_model, exclude=['ner'])

# Uso del programa
nlp = carregar_nlp('es_core_news_md')
doc = nlp("Esto es una prueba de carga automática.")

print(f"Texto procesado con éxito: {doc.text}")

La primera vez que ejecutes el programas verás que descarga el modelo:

```text
python3 programa-12.1-spa.py 
--- Inicializando el modelo  (es_core_news_md)...
--- El modelo 'es_core_news_md' no se ha encontrado.
Descargándolo automáticamente. Esto puede tardar un poco...
Collecting es-core-news-md==3.8.0
  Downloading [https://github.com/explosion/spacy-models/releases/download/es_core_news_md-3.8.0/es_core_news_md-3.8.0-py3-none-any.whl](https://github.com/explosion/spacy-models/releases/download/es_core_news_md-3.8.0/es_core_news_md-3.8.0-py3-none-any.whl) (42.3 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 24.9 MB/s eta 0:00:00
Installing collected packages: es-core-news-md
Successfully installed es-core-news-md-3.8.0
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
--- Carga completada. Inicializando (es_core_news_md)...
Texto procesado con éxito: Esto es una prueba de carga automática.
```

Pero si lo vuelves a ejecutar verás que ya no lo vuelve a descargar.

```text
python3 programa-12.1-spa.py 
--- Inicializando el modelo  (es_core_news_md)...
Texto procesado con éxito: Esto es una prueba de carga automática.
```

Fíjate que en el programa anterior se ha añadido `exclude=['ner']`. Esto sirve para no cargar el reconocedor de entidades nombradas (NER) si solo necesitas un análisis morfológico o sintáctico básico, ahorrando así tiempo de carga y memoria. 

Pero si los programas los haces para ti y ya sabes que has descargado el model vía terminal, solo necesitas hacer un simple `spacy.load()`.

### 1.3 Un primer programa de prueba

A continuación, haremos un programa que toma un texto con varias frases, incluyendo trampas como abreviaturas y números con puntos, para ver cómo se las arregla spaCy (programa-12.2-spa.py):

In [ ]:
import spacy

# Inicializamos el pipeline en castellano (asegúrate de haber hecho el python -m spacy download previamente)
nlp = spacy.load('es_core_news_md')

# El texto que queremos analizar
text = "spaCy es una herramienta fantástica para analizar el castellano."

# Procesamos el texto
doc = nlp(text)

# Comprobamos que funciona imprimiendo el texto procesado
print(doc.text)

# Mostramos por pantalla los diferentes análisis obtenidos
print(f"{'Texto':<12} | {'Lema':<12} | {'Categoría (POS)':<15} | {'Sintaxis (Dep)'}")
print("-" * 60)

for token in doc:
    # token.text: La palabra original
    # token.lemma_: El lema (forma base de la palabra)
    # token.pos_: La categoría gramatical (sustantivo, verbo, adjetivo...)
    # token.dep_: La función sintáctica dentro de la frase
    print(f"{token.text:<12} | {token.lemma_:<12} | {token.pos_:<15} | {token.dep_}")

A la salida tendremos algo como esto:

```text
python3 programa-12.2-spa.py 
spaCy es una herramienta fantástica para analizar el castellano.
Texto        | Lema         | Categoría (POS) | Sintaxis (Dep)
------------------------------------------------------------
spaCy        | spacy        | VERB            | nsubj
es           | ser          | AUX             | cop
una          | uno          | DET             | det
herramienta  | herramienta  | NOUN            | ROOT
fantástica   | fantástico   | ADJ             | amod
para         | para         | ADP             | mark
analizar     | analizar     | VERB            | acl
el           | el           | DET             | det
castellano   | castellano   | NOUN            | obj
.            | .            | PUNCT           | punct
```

Este programa ha sido solo un ejemplo. Durante este tutorial iremos profundizando en los diferentes niveles de análisis que ofrece esta librería.

**Ejercicio**: Modifica un poco el programa anterior, prueba otras oraciones u otros idiomas. Haz que la salida tenga la forma:

`spaCy|spaCy|PROPN es|ser|AUX una|un|DET herramienta|herramienta|NOUN fantástica|fantástico|ADJ para|para|ADP analizar|analizar|VERB el|el|DET castellano|castellano|NOUN .|.|PUNCT`

o bien, si quieres mantener la función sintáctica:

`spaCy|spaCy|PROPN|nsubj es|ser|AUX|cop una|un|DET|det herramienta|herramienta|NOUN|ROOT fantástica|fantástico|ADJ|amod para|para|ADP|mark analizar|analizar|VERB|advcl el|el|DET|det castellano|castellano|NOUN|obj .|.|PUNCT|punct`

### 1.4. Niveles de análisis en spaCy

spaCy puede ofrecer varios niveles de análisis y este tutorial se organiza a partir de estos niveles:

- Segmentación
- Tokenización
- Análisis morfológico
- Análisis morfosintáctico
- Análisis sintáctico

### 1.5. Idiomas y modelos disponibles en spaCy

En este enlace podréis explorar todos los idiomas y modelos disponibles en spaCy: [https://spacy.io/models](https://spacy.io/models)

### 2. Segmentación (Sentence Segmentation)

Cuando trabajamos con textos largos (como noticias, libros o páginas web enteras), el primer paso lógico para poder analizarlos correctamente es dividirlos en las frases individuales que los componen. Este proceso se conoce como segmentación o _Sentence Boundary Detection_. Hay que tener en cuenta que un segmento no siempre es una oración gramatical (por ejemplo, un título o un segmento que contenga solo expresiones numéricas) y muy a menudo en PLN se habla de segmentos en lugar de oraciones.

### 2.1. ¿Cómo funciona la segmentación en spaCy?

A diferencia de enfoques antiguos que solo buscaban puntos (.) o signos de interrogación (?) para cortar el texto, spaCy utiliza su modelo de aprendizaje profundo para entender la estructura gramatical. Esto le permite ser mucho más preciso y no confundirse con abreviaturas (como "Sr.", "Dr." o "p. ej.") ni con puntos suspensivos.

Cuando pasas un texto por el pipeline de spaCy (`doc = nlp(text)`), el sistema ya realiza esta división automáticamente gracias a su analizador sintáctico (o parser).

Puedes acceder a las frases detectadas a través de la propiedad `doc.sents`.

### 2.2. Programa de ejemplo: Extrayendo frases

A continuación, haremos un programa que toma un texto con varias frases, incluyendo trampas como abreviaturas y números con puntos, para ver cómo lo resuelve spaCy (programa-12.2-spa.py):

In [ ]:
import spacy

# 1. Cargamos el model en castellano
nlp = spacy.load('es_core_news_md')

# 2. Definimos un texto con varias frases y posibles trampas
text = """La Real Academia Española (RAE) fue fundada en 1713. ¡El Dr. Fernando Lázaro Carreter fue una figura clave en la historia de la institución! Hoy en día, casi 500 millones de personas lo hablan (apoximadamente). ¿Te gustaría aprender más?"""

# 3. Procesamos el texto
doc = nlp(text)

# 4. Iteramos sobre las frases y las imprimimos
print("--- FRASES DETECTADAS ---")
for i, frase in enumerate(doc.sents, 1):
    print(f"Frase {i}: {frase.text}")

Si lo ejecutas, verás que spaCy sabe perfectamente que "Dr." no es el final de una frase, cortando el texto exactamente donde toca. Te tiene que salir algo parecido a esto:

```text
python3 programa-12.2-spa.py 
--- FRASES DETECTADAS ---
Frase 1: La Real Academia Española (RAE) fue fundada en 1713.
Frase 2: ¡El Dr. Fernando Lázaro Carreter fue una figura clave en la historia de la institución!
Frase 3: Hoy en día, casi 500 millones de personas lo hablan (apoximadamente).
Frase 4: ¿Te gustaría aprender más?
``` 

**Ejercicio**: Pon a prueba el segmentador cambiando el texto de entrada por uno más complejo. Modifica el programa para que lea un archivo de texto y guarde el texto segmentado en otro archivo de texto.

### 2.3. Un detalle técnico importante: Contar frases

Si intentas hacer `len(doc.sents)` para saber cuántas frases hay en total, Python te dará un error. Esto ocurre porque doc.sents no es una lista normal de Python, sino un generador (un objeto que va calculando las frases una a una para ahorrar memoria). 

Si necesitas contar el número total de frases o acceder a una frase concreta (por ejemplo, la tercera frase), primero debes convertir este generador en una lista (programa-12.3-spa.py):

In [ ]:
import spacy

# 1. Cargamos el model en castellano
nlp = spacy.load('es_core_news_md')

# 2. Definimos un texto con varias frases y posibles trampas
text = """La Real Academia Española (RAE) fue fundada en 1713. ¡El Dr. Fernando Lázaro Carreter fue una figura clave en la historia de la institución! Hoy en día, casi 500 millones de personas lo hablan (apoximadamente). ¿Te gustaría aprender más?"""

# 3. Procesamos el texto
doc = nlp(text)

llista_frases = list(doc.sents)
# Ahora sí podemos contarlas
nombre_frases = len(llista_frases)
print(f"El texto tiene un total de {nombre_frases} frases.")

# Y acceder a una posición concreta (recuerda que empiezan por 0)
print(f"La segunda frase es: {llista_frases[1].text}")

Y en la salida tendremos:

```text
python3 programa-12.3-spa.py 
El texto tiene un total de 4 frases.
La segunda frase es: ¡El Dr. Fernando Lázaro Carreter fue una figura clave en la historia de la institución!
```

### 2.4. Personalización de la segmentación

Aunque el modelo por defecto de spaCy es muy robusto porque entiende la sintaxis de la frase, a veces trabajamos con textos de un dominio específico (medicina, derecho, documentos técnicos) que tienen formatos extraños o abreviaturas muy concretas.

En estos casos, spaCy nos permite personalizar la segmentación de diferentes maneras:

**A) Añadir nuevas abreviaturas**

Si el problema es que tenemos una abreviatura con un punto (por ejemplo, "Excmo." por "Excelentísimo") y spaCy se confunde pensando que es el final de la frase, la solución más fácil es añadir esta palabra a las excepciones de su tokenizador. Si el tokenizador entiende que "Excmo." es una sola pieza, el segmentador de frases ya no cortará por ese punto.

Veamos cómo hacerlo en el programa programa-12.5-spa.py:

In [ ]:
import spacy
from spacy.symbols import ORTH

nlp = spacy.load('es_core_news_md')

text = "El Excmo. Sr. Pérez está de viaje. Volverá pronto."
doc = nlp(text)

print("--- SEGMENTACIÓN POR DEFECTO ---")
for frase in doc.sents:
    print(frase.text)

excepcion = [{ORTH: "Excmo."}]
nlp.tokenizer.add_special_case("Excmo.", excepcion)

doc = nlp(text)

print("--- SEGMENTACIÓN AÑADIENDO ABREVIATURA ---")
for frase in doc.sents:
    print(frase.text)

Que nos dará como resultado:

```text
--- SEGMENTACIÓN POR DEFECTO ---
El Excmo.
Sr. Pérez está de viaje.
Volverá pronto.
--- SEGMENTACIÓN AÑADIENDO ABREVIATURA ---
El Excmo. Sr. Pérez está de viaje.
Volverá pronto.
```

**B) Crear reglas personalizadas de segmentación**

A veces queremos forzar el corte de una frase basándonos en caracteres concretos, como un salto de línea (\n) o un punto y coma (;). Para hacerlo, podemos crear un componente personalizado y añadirlo al pipeline de spaCy justo antes del parser ( que es quien normalmente decide dónde acaban las frases).

Para forzar o prohibir el corte de una frase antes de una palabra concreta, utilizamos el atributo is_sent_start (inicio de frase) de cada token. (programa-12.6-spa.py)

In [ ]:
import spacy
from spacy.language import Language

nlp = spacy.load('es_core_news_md')

# 1. Definimos nuestro componente de reglas
@Language.component("reglas_segmentacion_custom")
def segmentacio_custom(doc):
    # Iteramos por todos los tokens menos el último
    for token in doc[:-1]: 
        # Si encontramos un punto y coma o un salto de línea...
        if token.text in [";", "\n"]:
            # Marcamos el siguiente token como el inicio de una nueva frase
            doc[token.i + 1].is_sent_start = True
    return doc

# 2. Añadimos el componente al pipeline, antes del parser
nlp.add_pipe("reglas_segmentacion_custom", before="parser")

# 3. Lo probamos
text = "Primera frase normal. Segunda frase con punto y coma; tercera frase forzada por la regla.\nY cuarta frase después del salto de línea."
doc = nlp(text)

print("--- SEGMENTACIÓN CON REGLAS CUSTOM ---")
for frase in doc.sents:
    # Utilizamos .strip() para limpiar espacios o saltos de línea al principio/final
    print(f"- {frase.text.strip()}")

## 3. Tokenización

Una vez que tenemos el texto dividido en frases, el siguiente paso es trocear esta cadena de caracteres en las unidades mínimas de significado: las palabras y los signos de puntuación. Estas piezas individuales se denominan tokens, y el proceso de separarlas es la tokenización.

### 3.1 ¿Cómo funciona la tokenización en spaCy?

A primera vista, tokenizar podría parecer tan sencillo como cortar el texto cada vez que hay un espacio en blanco. Pero en lenguas como el castellano, la realidad es un poco más compleja:

- Puntuación: El punto final, las comas o los signos de interrogación van pegados a la palabra, pero se tienen que separar porque son elementos sintácticos independientes. En castellano, además, contamos con los signos de apertura (¿, ¡).

La ventaja de utilizar el modelo en castellano de spaCy (es_core_news_md) es que ya incorpora todas las reglas y excepciones propias de este idioma, siguiendo los estándares lingüísticos de Universal Dependencies. Además, spaCy realiza una tokenización "no destructiva": nunca elimina los espacios originales ni modifica el texto fuente, solo crea una lista de tokens que apuntan a las diferentes partes del texto.

### 3.2. Programa de ejemplo: Explorando los tokens

A continuación veremos cómo spaCy trocea una frase que contiene algunas de estas dificultades típicas del castellano (enclíticos, contracciones y signos de puntuación dobles) (programa-12.8-spa.py):

In [ ]:
import spacy

# 1. Cargamos el modelo en castellano
nlp = spacy.load('es_core_news_md')

# 2. Texto con pronombres enclíticos, contracciones y puntuación
text = "¿El estudiante decidió apuntarse al curso de la RAE, verdad?"

# 3. Procesamos el texto
doc = nlp(text)

# 4. Imprimimos una cabecera para verlo claro
print(f"{'Índice':<6} | {'Token':<12} | {'¿Es letra?':<12} | {'¿Es puntuación?'}")
print("-" * 55)

# 5. Iteramos sobre cada token del objeto 'doc'
for token in doc:
    # token.i : el índice (posición) del token dentro de la frase
    # token.text : el texto original del token
    # token.is_alpha : devuelve True si solo contiene letras
    # token.is_punct : devuelve True si es un signo de puntuación
    
    print(f"{token.i:<6} | {token.text:<12} | {str(token.is_alpha):<12} | {str(token.is_punct)}")

Si ejecutas el programa anterior, la salida por pantalla te mostrará exactamente cómo el modelo ha tomado las decisiones de separación:

```text
Índice | Token        | ¿Es letra?   | ¿Es puntuación?
-------------------------------------------------------
0      | ¿            | False        | True
1      | El           | True         | False
2      | estudiante   | True         | False
3      | decidió      | True         | False
4      | apuntarse    | True         | False
5      | al           | True         | False
6      | curso        | True         | False
7      | de           | True         | False
8      | la           | True         | False
9      | RAE          | True         | False
10     | ,            | False        | True
11     | verdad       | True         | False
12     | ?            | False        | True
```

Fíjate en los detalles importantes:

- Ha aislado correctamente los interrogantes, tanto de apertura como de cierre.
- Mantiene el verbo con el pronombre enclítico en `apuntarse` según el estándar del modelo.
- También ha aislado el signo de puntuación `,`.

## 4. Análisis morfológico

Cuando empezamos a analizar las palabras de un texto para clasificarlas (saber si son sustantivos, verbos, adjetivos, etc.), es fundamental entender la diferencia entre dos conceptos clave en el Procesamiento del Lenguaje Natural (PLN):

- El análisis morfológico (sin desambiguar): Consiste en analizar una palabra de manera aislada, fuera de contexto, y listar todas sus posibles categorías y formas. Por ejemplo, en castellano, la palabra "sobre" puede ser un sustantivo masculino (un sobre de papel), una preposición (sobre la mesa) o una forma del verbo sobrar (espero que sobre comida).
- El análisis morfosintáctico (desambiguado o POS Tagging): Consiste en analizar la palabra dentro de su contexto (la frase) para determinar cuál de estas opciones es la única correcta en esa oración específica.

### 4.1. El enfoque de spaCy vs. Diccionarios tradicionales

Herramientas históricas y muy potentes como [FreeLing](https://nlp.lsi.upc.edu/freeling/node/1) funcionan en dos pasos: primero consultan un diccionario gigantesco para generar todas las opciones posibles (análisis morfológico) y, después, aplican reglas o modelos estadísticos para descartar las incorrectas (desambiguación morfosintáctica).

spaCy funciona de manera diferente. Al estar construido sobre redes neuronales profundas, no pierde el tiempo calculando todas las opciones posibles de una palabra aislada. spaCy lee toda la frase a la vez y predice directamente la respuesta morfosintáctica correcta.

Por lo tanto, spaCy no realiza un análisis morfológico puro. ¿Cómo lo podemos solucionar si en nuestro proyecto necesitamos ver todas las posibilidades teóricas de una palabra? La solución es combinar la potencia de tokenización de spaCy con los diccionarios abiertos de FreeLing.

### 4.2. Las etiquetas de FreeLing (Formato EAGLES)

Los diccionarios de FreeLing son archivos de texto plano donde cada línea contiene tres elementos: la palabra flexionada (la forma), su lema (la raíz) y una etiqueta morfológica.

Estas etiquetas siguen el estándar europeo [EAGLES](https://freeling-user-manual.readthedocs.io/en/latest/tagsets/), que es un sistema posicional. Cada letra de la etiqueta nos aporta una información concreta. Por ejemplo, la etiqueta NCMS000:

- N: Categoría principal (Nombre / Sustantivo)
- C: Tipo (Común)
- M: Género (Masculino)
- S: Número (Singular)

Si miramos un verbo sencillo como VMIP1S0 (canto):

- V: Categoría principal (Verbo)
- M: Tipo (Principal / Main)
- I: Modo (Indicativo)
- P: Tiempo (Presente)
- 1: Persona (Primera)
- S: Número (Singular)

(Nota: El último cero 0 en los verbos indica que no tienen marca de género, a diferencia de los participios, donde sí encontraríamos una M o una F).

En el siguiente programa (programa-12.9-spa.py), uniremos lo mejor de ambos mundos. Utilizaremos spaCy exclusivamente para segmentar y cortar la oración correctamente (aislando los signos de puntuación). Una vez que tengamos los tokens limpios, buscaremos cada palabra en el diccionario freeling-spa.txt para extraer todas sus interpretaciones morfológicas posibles, simulando así un analizador morfológico puro.

In [ ]:
import spacy
import os

def cargar_diccionario_morfologico(ruta_fichero):
    """
    Lee el diccionario de FreeLing y devuelve uno de Python.
    """
    diccionario = {}
    if not os.path.exists(ruta_fichero):
        print(f"Error: No se ha encontrado el fichero '{ruta_fichero}'.")
        return diccionario

    print(f"Cargando el diccionario '{ruta_fichero}' en la memoria...")
    with open(ruta_fichero, 'r', encoding='utf-8') as fichero:
        for linea in fichero:
            parts = linea.strip().split()
            if len(parts) >= 3:
                forma, lema, etiqueta = parts[0], parts[1], parts[2]
                interpretacion = f"[{lema} - {etiqueta}]"
                
                if forma not in diccionario:
                    diccionario[forma] = []
                diccionario[forma].append(interpretacion)
                
    print(f"¡Diccionario cargado! Contiene {len(diccionario)} formas únicas.\n")
    return diccionario


# 1. Cargamos el modelo de spaCy en castellano (solo para tokenizar)
print("--- Inicializando spaCy...")
try:
    nlp = spacy.load('es_core_news_md')
except OSError:
    print("Descargando el modelo es_core_news_md...")
    from spacy.cli import download
    download('es_core_news_md')
    nlp = spacy.load('es_core_news_md')

# 2. Cargamos el diccionario externo
fichero_diccionario = 'freeling-spa.txt'
analizador_morfologico = cargar_diccionario_morfologico(fichero_diccionario)

# 3. Solo ejecutamos el análisis si el diccionario se ha cargado correctamente
if analizador_morfologico:
    
    # Definimos la oración de entrada con la palabra polisémica "sobre"
    oracion = "¿El estudiante dejó el sobre amarillo sobre la mesa, verdad?"
    
    # Tokenizamos la oración con spaCy
    doc = nlp(oracion)
    
    print(f"Oración a analizar: {oracion}\n")
    print(f"{'Token':<12} | {'Interpretaciones Posibles'}")
    print("-" * 80)
    
    # 4. Iteramos sobre los tokens que ha separado spaCy
    for token in doc:
        # Convertimos a minúsculas para buscarlo en el diccionario
        palabra_busqueda = token.text.lower()
        
        # Caso A: La palabra está en el diccionario de FreeLing
        if palabra_busqueda in analizador_morfologico:
            opciones = ", ".join(analizador_morfologico[palabra_busqueda])
            print(f"{token.text:<12} | {opciones}")
            
        # Caso B: Es un signo de puntuación (lo sabemos gracias a spaCy)
        elif token.is_punct:
            print(f"{token.text:<12} | [{token.text} - PUNCT]")
            
        # Caso C: Son espacios en blanco (los ignoramos)
        elif token.is_space:
            continue
            
        # Caso D: La palabra no está en el diccionario (neologismo, error, etc.)
        else:
            print(f"{token.text:<12} | DESCONOCIDA (No está en el diccionario)")

### 4.3. El problema de la ambigüedad

Si ejecutas este programa, verás de inmediato el gran reto del procesamiento del lenguaje. Cuando el código llega a la palabra "sobre", imprime todas las opciones del diccionario (como sustantivo, como preposición y como forma del verbo sobrar). Solo con el diccionario, el ordenador es incapaz de saber cuál de estas funciones está realizando el "sobre" amarillo o el "sobre" la mesa de nuestra frase.

```text
--- Inicializando spaCy...
Cargando el diccionario 'freeling-spa.txt' en la memoria...
¡Diccionario cargado! Contiene 556428 formas únicas.

Oración a analizar: ¿El estudiante dejó el sobre amarillo sobre la mesa, verdad?

Token        | Interpretaciones Posibles
--------------------------------------------------------------------------------
¿            | [¿ - PUNCT]
El           | [el - DA0MS0]
estudiante   | [estudiante - AQ0CS00], [estudiante - NCCS000]
dejó         | [dejar - VMIS3S0]
el           | [el - DA0MS0]
sobre        | [sobre - NCMS000], [sobre - SP], [sobrar - VMM03S0], [sobrar - VMSP1S0], [sobrar - VMSP3S0]
amarillo     | [amarillo - AQ0MS00], [amarillo - NCMS000]
sobre        | [sobre - NCMS000], [sobre - SP], [sobrar - VMM03S0], [sobrar - VMSP1S0], [sobrar - VMSP3S0]
la           | [la - NCMS000], [el - DA0FS0], [lo - PP3FSA0]
mesa         | [mesa - NCFS000], [mesar - VMIP3S0], [mesar - VMM02S0]
,            | [, - PUNCT]
verdad       | [verdad - NCFS000]
?            | [? - PUNCT]
```

Para resolver este problema, necesitamos pasar al siguiente nivel.

## 5. Análisis Morfosintáctico (POS Tagging)

En el apartado anterior hemos visto el gran problema de utilizar solo un diccionario: la ambigüedad. Cuando veíamos la palabra "sobre", el sistema nos daba una lista de opciones (sustantivo, preposición, verbo...), pero no sabía cuál era la correcta.

En la vida real, cuando hablamos, una palabra solo tiene una única función dentro de una frase concreta. Identificar cuál es esa función exacta, dependiendo de las palabras de alrededor, es lo que llamamos análisis morfosintáctico o POS Tagging (Part-of-Speech Tagging).

### 5.1. ¿Cómo desambigua spaCy?

Aquí es donde brilla la inteligencia artificial. spaCy no lee las palabras de una en una, sino que utiliza redes neuronales para leer la frase entera de golpe. Al ver que detrás del primer "sobre" hay un adjetivo ("amarillo"), la red deduce matemáticamente que eso tiene que ser un sustantivo. En cambio, al ver que el segundo "sobre" va seguido de un artículo y un sustantivo ("la mesa"), deduce que es una preposición de espacio.

### 5.2. El estándar Universal Dependencies (UD)

Mientras que FreeLing utiliza el estándar europeo EAGLES, spaCy utiliza el estándar internacional Universal Dependencies (UD). Este sistema está diseñado para que las categorías gramaticales sean idénticas en más de 100 idiomas diferentes.

En lugar de una sola etiqueta combinada (como NCMS000), spaCy separa la información en dos atributos:

- `token.pos_` (Categoría principal): Nos dice si es un sustantivo (NOUN), un verbo (VERB), un adjetivo (ADJ), una preposición (ADP), etc.
- `token.morph` (Rasgos morfológicos): Nos detalla la flexión de la palabra (género, número, tiempo, persona...). Por ejemplo: Gender=Masc|Number=Sing.

En este programa (programa-12.10-spa.py) haremos pasar la misma frase trampa por el analizador de spaCy para ver cómo resuelve la ambigüedad.

In [ ]:
import spacy

# 1. Cargamos el modelo en castellano
print("--- Inicializando spaCy...")
try:
    nlp = spacy.load('es_core_news_md')
except OSError:
    from spacy.cli import download
    download('es_core_news_md')
    nlp = spacy.load('es_core_news_md')

# 2. Nuestra oración con la palabra polisémica "sobre"
oracio = "¿El estudiante dejó el sobre amarillo sobre la mesa, verdad?"

# 3. Procesamos la oración (aquí es donde se hace la magia de la desambiguación)
doc = nlp(oracio)

print(f"\nOración analizadora: {oracio}\n")
print(f"{'Token':<12} | {'POS (Categoría)':<15} | {'Morfología (Rasgos)'}")
print("-" * 75)

# 4. Iteramos sobre cada token para extraer la etiqueta correcta
for token in doc:
    # token.text : La palabra original
    # token.pos_ : La categoría gramatical desambiguada (NOUN, VERB, ADP...)
    # token.morph: Los rasgos morfológicos (Género, Número...)
    
    # Convertimos el objeto morph a texto (string) para poder imprimirlo bien. 
    # Si está vacío (ej. preposiciones), ponemos un guion.
    morfologia = str(token.morph) if str(token.morph) else "-"
    
    print(f"{token.text:<12} | {token.pos_:<15} | {morfologia}")

Si ejecutas el programa, te llevarás una sorpresa. La salida por pantalla muestra un claro error de análisis por parte del modelo:

```text
--- Inicializando spaCy...

Oración analizadora: ¿El estudiante dejó el sobre amarillo sobre la mesa, verdad?

Token        | POS (Categoría) | Morfología (Rasgos)
---------------------------------------------------------------------------
¿            | PUNCT           | PunctSide=Ini|PunctType=Qest
El           | DET             | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
estudiante   | NOUN            | Number=Sing
dejó         | VERB            | Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin
el           | DET             | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
sobre        | ADP             | -
amarillo     | NOUN            | Gender=Masc|Number=Sing
sobre        | ADP             | -
la           | DET             | Definite=Def|Gender=Fem|Number=Sing|PronType=Art
mesa         | NOUN            | Gender=Fem|Number=Sing
,            | PUNCT           | PunctType=Comm
verdad       | NOUN            | Gender=Fem|Number=Sing
?            | PUNCT           | PunctSide=Fin|PunctType=Qest
```

¡El modelo se ha equivocado con el primer "sobre"! En lugar de etiquetarlo como sustantivo (NOUN), lo ha clasificado como preposición (ADP). Al cometer este error, se ha visto obligado a etiquetar la palabra amarillo como sustantivo (NOUN) para intentar buscarle un sentido matemático a la estructura.

Esto ocurre porque spaCy se basa en modelos estadísticos y redes neuronales probabilísticas. En los textos de entrenamiento del castellano, la palabra sobre aparece de forma masiva como preposición ("hablar sobre política", "sobre el tejado"). Ante esta fuerte tendencia estadística, el modelo (es_core_news_md) ha flaqueado y no ha sido capaz de desambiguarla correctamente en este contexto concreto.

En el Procesamiento del Lenguaje Natural, los errores de desambiguación en palabras con una categoría gramatical muy dominante (como sobre) son habituales. Para mitigar estos errores en entornos de producción, se suele recurrir a modelos más grandes (como es_core_news_lg o modelos basados en Transformers como BERT) o bien se introducen reglas de corrección posteriores en el pipeline.

**Ejercicio:** Si cambias el modelo a es_core_news_lg, ¿continua produciéndose el error?

**El siguiente nivel: Los modelos Transformer (_trf)**

Los modelos que has probado (_md y _lg) utilizan una arquitectura clásica de spaCy basada en redes neuronales convolucionales (CNN) y vectores estáticos. Son muy rápidos y ligeros, pero tienen menos "luces" para entender contextos sutiles o tramposos.

Los modelos Transformer integran arquitecturas como BERT o RoBERTa. En lugar de asignar probabilidades rígidas, calculan representaciones contextuales profundas: la definición de cada palabra cambia dinámicamente según todas las demás palabras de la frase.

El modelo oficial: es_dep_news_trf

spaCy ofrece un modelo oficial basado en Transformers para el español. Está entrenado sobre una variante de BERT en español (habitualmente MarIA o BETO).

Para probarlo en tu código, primero debes instalar la extensión de spaCy para Transformers y descargar el modelo desde la terminal:

```bash
pip install spacy[transformers]
python -m spacy download es_dep_news_trf
```

Y después cambias el modelo en el script a es_dep_news_trf. Ahora, si vuelves a ejecutar el programa, tendría que analizar bien:

```text
--- Inicializando spaCy...

Oración analizadora: ¿El estudiante dejó el sobre amarillo sobre la mesa, verdad?

Token        | POS (Categoría) | Morfología (Rasgos)
---------------------------------------------------------------------------
¿            | PUNCT           | PunctSide=Ini|PunctType=Qest
El           | DET             | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
estudiante   | NOUN            | Number=Sing
dejó         | VERB            | Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin
el           | DET             | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
sobre        | NOUN            | Gender=Masc|Number=Sing
amarillo     | ADJ             | Gender=Masc|Number=Sing
sobre        | ADP             | -
la           | DET             | Definite=Def|Gender=Fem|Number=Sing|PronType=Art
mesa         | NOUN            | Gender=Fem|Number=Sing
,            | PUNCT           | PunctType=Comm
verdad       | NOUN            | Gender=Fem|Number=Sing
?            | PUNCT           | PunctSide=Fin|PunctType=Qest
```

### 5.3. La Lematización: Buscando la forma de diccionario

Una vez que spaCy ha analizado la frase, puede dar un paso más de manera muy precisa: la lematización.

Lematizar consiste en extraer el "lema" o forma base de una palabra, que es exactamente la forma bajo la cual la buscaríamos en un diccionario:

- Para los verbos, el lema es el infinitivo (p. ej., dejó -> dejar, canto -> cantar).
- Para los sustantivos y adjetivos, el lema suele ser el masculino singular (p. ej., mesas -> mesa, amarillo -> amarillo).

spaCy nos proporciona esta información de manera muy sencilla a través del atributo `token.lemma_`.

En este programa (programa-12.11-spa.py) haremos pasar nuestra oración de prueba para ver cómo extrae la raíz de cada palabra basándose en el análisis morfosintáctico previo.

In [ ]:
import spacy

# 1. Cargamos el modelo en castellano
print("--- Inicializando spaCy...")
try:
    nlp = spacy.load('es_core_news_md')
except OSError:
    from spacy.cli import download
    download('es_core_news_md')
    nlp = spacy.load('es_core_news_md')

# 2. Nuestra oración con la palabra polisémica "sobre"
oracio = "¿El estudiante dejó el sobre amarillo sobre la mesa, verdad?"

# 3. Procesamos la oración (se realiza el análisis y la lematización)
doc = nlp(oracio)

print(f"\nOración analizada: {oracio}\n")
print(f"{'Token':<12} | {'Lema':<12} | {'POS':<6} | {'Morfología (Rasgos)'}")
print("-" * 80)

# 4. Iteramos sobre cada token para extraer toda la información
for token in doc:
    # token.text    : La palabra original
    # token.lemma_ : La forma base (lema)
    # token.pos_    : La categoría gramatical
    # token.morph   : Los rasgos morfológicos
    
    morfologia = str(token.morph) if str(token.morph) else "-"
    
    print(f"{token.text:<12} | {token.lemma_:<12} | {token.pos_:<6} | {morfologia}")

Si ejecutas el programa, la salida por pantalla será la siguiente:

```text
--- Inicializando spaCy...

Oración analizada: ¿El estudiante dejó el sobre amarillo sobre la mesa, verdad?

Token        | Lema         | POS    | Morfología (Rasgos)
--------------------------------------------------------------------------------
¿            | ¿            | PUNCT  | PunctSide=Ini|PunctType=Qest
El           | el           | DET    | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
estudiante   | estudiante   | NOUN   | Number=Sing
dejó         | dejar        | VERB   | Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin
el           | el           | DET    | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
sobre        | sobre        | ADP    | -
amarillo     | amarillo     | NOUN   | Gender=Masc|Number=Sing
sobre        | sobre        | ADP    | -
la           | el           | DET    | Definite=Def|Gender=Fem|Number=Sing|PronType=Art
mesa         | mesa         | NOUN   | Gender=Fem|Number=Sing
,            | ,            | PUNCT  | PunctType=Comm
verdad       | verdad       | NOUN   | Gender=Fem|Number=Sing
?            | ?            | PUNCT  | PunctSide=Fin|PunctType=Qest
```

Fíjate en cómo influye el análisis en los resultados:

- El impacto del error previo: Como el modelo es_core_news_md se ha confundido arrastrado por la estadística y ha etiquetado el primer sobre como preposición (ADP), su lema se queda simplemente como sobre. Además, al haber etiquetado amarillo como un sustantivo (NOUN), su lema es amarillo. Cambia de modelo para ver si se soluciona este problema y lematiza bien.
- Lematización verbal correcta: El modelo detecta sin problemas que la forma conjugada dejó proviene del infinitivo dejar.
- Artículos: Sabe perfectamente que tanto El como la tienen como lema común la forma base del artículo masculino singular: el.

## 6. Análisis sintáctico

Una vez sabemos qué es cada palabra (morfosintaxis) y de dónde viene (lema), el siguiente paso es entender cómo se relacionan las palabras entre sí para formar frases. Esto es el análisis sintáctico. En el mundo del PLN, existen dos formas principales de representar esta estructura:

- Análisis de dependencias (spaCy): Se basa en flechas que conectan palabras. Una palabra depende de otra (ej.: el adjetivo depende del sustantivo). Es un modelo muy rápido e ideal para la extracción de datos.
- Análisis de constituyentes (Benepar): Se basa en la jerarquía de sintagmas, es el clásico análisis sintáctico en árbol. Es un modelo más lento pero muy útil para estudios gramaticales profundos.

### 6.1. Análisis de dependencias con spaCy

spaCy identifica qué palabra es el "head" y cuál es el "dependiente". La etiqueta de la relación (sujeto, objeto, etc.) se encuentra en el atributo `token.dep_`. (programa-12.12-spa.py)

In [ ]:
import spacy

nlp = spacy.load('es_core_news_md')
doc = nlp("El estudiante lee un libro muy antiguo.")

print(f"{'Token':<12} | {'Relación':<10} | {'Cabeza (Padre)':<12}")
print("-" * 42)
for token in doc:
    print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12}")

Que proporciona el siguiente resultado:

```text
Token        | Relación   | Head        
------------------------------------------
El           | det        | estudiante  
estudiante   | nsubj      | lee         
lee          | ROOT       | lee         
un           | det        | libro       
libro        | obj        | lee         
muy          | advmod     | antiguo     
antiguo      | amod       | libro       
.            | punct      | lee         
```

Nota: La palabra principal de la frase, que no depende de ninguna otra, recibe la etiqueta ROOT (normalmente el verbo principal).

### 6.2. Análisis de sintagmes (La agrupación de palabras)

Aunque spaCy no realiza un análisis de "constituyentes" completo (aquel que introduce toda la frase dentro de cajas jerárquicas), sí que nos permite extraer de manera muy eficiente los Sintagmes Nominales (Noun Chunks).

Estos son los grupos de palabras que tienen un sustantivo como núcleo (incluyendo sus determinantes y adjetivos).

Programa de ejemplo (Sintagmes Nominales) (programa-12.13-spa.py)

In [ ]:
import spacy

nlp = spacy.load('es_core_news_md')
doc = nlp("El estudiante de la universidad lee un libro muy antiguo.")

print("--- Sintagmas Nominales detectados ---")
for chunk in doc.noun_chunks:
    # Mostramos el texto del sintagma, su núcleo y su función
    print(f"Sintagma: {chunk.text:<30} | Núcleo: {chunk.root.text}")

Que proporciona la siguiente salida:

```text
--- Sintagmes Nominales detectados ---
Sintagma: El estudiante                  | Núcleo: estudiante
Sintagma: la universidad                 | Núcleo: universidad
Sintagma: un libro muy antiguo           | Núcleo: libro
```

Visualización gráfica (displaCy)

spaCy incluye un motor visual que nos permite generar un diagrama de flechas (programa-12.14-spa.py).

In [ ]:
import spacy
from spacy import displacy
from pathlib import Path

nlp = spacy.load('es_core_news_md')
doc = nlp("El estudiante de la universidad lee un libro muy antiguo.")

# Renderizamos el gráfico directamente dentro del Jupyter Notebook
displacy.render(doc, style="dep", jupyter=True)

# Opcionalmente también lo guardamos en un archivo HTML para abrirlo en el navegador
svg = displacy.render(doc, style="dep", jupyter=False)
Path("arbol_sintactico.html").write_text(svg, encoding="utf-8")
print("\nTambién se ha guardado el archivo 'arbol_sintactico.html' en el directorio local.")

Recuerda que el gráfico se ha generado en el archivo html `arbol_sintactico.html`. Para visualizarlo lo puedes abrir con cualquier navegador. Tendrá el siguiente aspecto:

<img width="2212" height="300" alt="image" src="https://github.com/user-attachments/assets/309175c2-1abb-44f9-97db-fb0708868202" />

## 7. Otros tipos de análisis y aplicaciones

Hasta aquí hemos visto cómo podemos implementar los diferentes niveles de análisis en spaCy. A continuación, os propongo algunos análisis alternativos y algunas aplicaciones concretas de los análisis que hemos visto hasta ahora. Explora al menos una de estas secciones. Pueden servirte de inspiración para el proyecto de programación que tendrás que desarrollar más adelante.

- [Complemento A: Reconocimiento de Entidades Nominales (NER)](https://github.com/aoliverg/python/wiki/Complement-A:-Reconeixement-d'Entitats-Nominals-(NER))
- [Complemento B: Análisis de sentimiento (Sentiment Analysis)](https://github.com/aoliverg/python/wiki/An%C3%A1lisis-de-sentimiento-(Sentiment-Analysis))